# MeteorNet - Google Colab Runner

This notebook runs the original codebase in Colab while keeping the same project scripts and behavior (`train.py`, `evaluate.py`, `predict.py`).

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Nikhil306-ctrl/-MeteorNet-Real-Time-Meteor-Detection.git"
PROJECT_DIR = Path("/content/-MeteorNet-Real-Time-Meteor-Detection")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print(f"Working directory: {Path.cwd()}")

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Optional: mount Google Drive if your dataset/checkpoints are stored there.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Update these paths as needed.
DATA_DIR = "/content/drive/MyDrive/D_2 DATASET (METEORS)/sum_images"
MODEL_TYPE = "meteor"  # "meteor" or "resnet"
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
LIMIT_PER_CLASS = None  # e.g., 500 for quick smoke runs
USE_PRETRAINED = False  # set True for resnet pretrained weights

In [ ]:
def run_command(command):
    print("Running:", " ".join(command))
    subprocess.run(command, check=True)

train_command = [
    sys.executable, "train.py",
    "--data", DATA_DIR,
    "--model", MODEL_TYPE,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE)
]

if LIMIT_PER_CLASS is not None:
    train_command.extend(["--limit-per-class", str(LIMIT_PER_CLASS)])

if USE_PRETRAINED:
    train_command.append("--pretrained")

run_command(train_command)

In [ ]:
checkpoint_path = f"models/{MODEL_TYPE}_best.pth"

evaluate_command = [
    sys.executable, "evaluate.py",
    "--data", DATA_DIR,
    "--model", MODEL_TYPE,
    "--checkpoint", checkpoint_path,
    "--batch-size", str(BATCH_SIZE)
]

if LIMIT_PER_CLASS is not None:
    evaluate_command.extend(["--limit-per-class", str(LIMIT_PER_CLASS)])

run_command(evaluate_command)

In [ ]:
from pathlib import Path

meteor_dir = Path(DATA_DIR) / "meteor"
candidate_images = sorted(
    list(meteor_dir.glob("*.jpg"))
    + list(meteor_dir.glob("*.jpeg"))
    + list(meteor_dir.glob("*.png"))
    + list(meteor_dir.glob("*.bmp"))
)

if not candidate_images:
    raise FileNotFoundError(f"No prediction image found in {meteor_dir}")

sample_image = str(candidate_images[0])
print("Using sample image:", sample_image)

predict_command = [
    sys.executable, "predict.py",
    "--image", sample_image,
    "--checkpoint", checkpoint_path
]

run_command(predict_command)